In [ ]:
# !pip install --upgrade pip

# # ember github for reference 
!git clone https://github.com/FutureComputing4AI/EMBER2024.git
%pip install ./EMBER2024

%pip install pandas
%pip install altair

# # thrember dependencies
!pip uninstall -y signify
%pip install "signify==0.7.1"

# might need libomp installed:
!brew install libomp

In [ ]:
# imports
import os
from pathlib import Path
import thrember
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import lightgbm as lgb
import polars as pl
import altair as alt
from sklearn.metrics import roc_auc_score, roc_curve
alt.renderers.enable('default')

In [ ]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"
train_df = pd.read_parquet(DATA_DIR / "win32_detection_train_20pct.parquet")

print("rows:", len(train_df))
print("duplicate rows:", train_df.duplicated().sum())
print("duplicate sha256:", train_df["sha256"].duplicated().sum())


print("Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

display(train_df.head())


print("\nData types:")
print(train_df.dtypes)

print(train_df.iloc[2].strings)
print(train_df.iloc[2].general)



In [ ]:
label_counts = (
    train_df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

label_counts["category"] = label_counts["label"].map({
    0: "Benign",
    1: "Malware"
})

chart = alt.Chart(label_counts).mark_bar().encode(
    x=alt.X("category:N", title="File Classification"),
    y=alt.Y("count:Q", title="Number of Samples"),
    tooltip=["category", "count"]
).properties(
    title="Distribution of Malware and Benign Samples",
    width=500,
    height=350
)





chart

In [ ]:
#First inspect one imports value:
import json

sample_imports = train_df.iloc[0]["imports"]

if isinstance(sample_imports, str):
    sample_imports = json.loads(sample_imports)

print(type(sample_imports))
print(sample_imports)

In [ ]:
#Number of imported APIs per file
import json
import pandas as pd
import altair as alt

def parse_imports(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}


def count_imported_apis(import_data):
    if isinstance(import_data, dict):
        total = 0

        for apis in import_data.values():
            if isinstance(apis, list):
                total += len(apis)

        return total

    if isinstance(import_data, list):
        return len(import_data)

    return 0


parsed_imports = train_df["imports"].apply(parse_imports)

api_count_df = pd.DataFrame({
    "api_count": parsed_imports.apply(count_imported_apis),
    "label": train_df["label"]
})

api_count_df["category"] = api_count_df["label"].map({
    0: "Benign",
    1: "Malware"
})

display(
    api_count_df.groupby("category")["api_count"]
    .agg(["count", "mean", "median", "std", "max"])
    .round(2)
)

In [ ]:
#Because the dataset is large, aggregate the histogram before sending it to Altair:
import numpy as np

upper_limit = api_count_df["api_count"].quantile(0.99)

api_plot_df = api_count_df[
    api_count_df["api_count"] <= upper_limit
].copy()

bin_edges = np.linspace(
    api_plot_df["api_count"].min(),
    api_plot_df["api_count"].max(),
    41
)

api_plot_df["api_bin"] = pd.cut(
    api_plot_df["api_count"],
    bins=bin_edges,
    include_lowest=True
)

api_histogram_df = (
    api_plot_df
    .groupby(["category", "api_bin"], observed=True)
    .size()
    .reset_index(name="count")
)

api_histogram_df["bin_midpoint"] = (
    api_histogram_df["api_bin"]
    .apply(lambda interval: interval.mid)
    .astype(float)
)

api_histogram_df["percentage"] = (
    api_histogram_df.groupby("category")["count"]
    .transform(lambda values: values / values.sum() * 100)
)

api_histogram_plot = api_histogram_df[
    ["category", "bin_midpoint", "count", "percentage"]
].copy()

In [ ]:
#Plot it:
alt.Chart(api_histogram_plot).mark_line(
    point=True
).encode(
    x=alt.X(
        "bin_midpoint:Q",
        title="Number of imported APIs"
    ),
    y=alt.Y(
        "percentage:Q",
        title="Percentage of files"
    ),
    color=alt.Color(
        "category:N",
        title="File type"
    ),
    tooltip=[
        "category:N",
        alt.Tooltip(
            "bin_midpoint:Q",
            title="Imported APIs",
            format=".0f"
        ),
        alt.Tooltip(
            "percentage:Q",
            title="Percentage",
            format=".2f"
        )
    ]
).properties(
    title="Imported API Count: Malware vs. Benign",
    width=650,
    height=400
)

In [ ]:
"""Flatten the imported API names

This converts each file’s nested imports into one list:"""
def flatten_imported_apis(import_data):
    api_names = []

    if isinstance(import_data, dict):
        for apis in import_data.values():
            if isinstance(apis, list):
                api_names.extend(
                    str(api).lower()
                    for api in apis
                )

    elif isinstance(import_data, list):
        api_names.extend(
            str(api).lower()
            for api in import_data
        )

    return api_names


api_lists = parsed_imports.apply(flatten_imported_apis)

print(api_lists.iloc[0][:20])

In [ ]:
"""Compare networking API usage

Define a reasonable set of networking-related terms:"""

network_terms = [
    "socket",
    "connect",
    "send",
    "recv",
    "bind",
    "listen",
    "accept",
    "internetopen",
    "internetconnect",
    "internetreadfile",
    "internetwritefile",
    "httpopenrequest",
    "httpsendrequest",
    "urlopen",
    "urldownloadtofile",
    "winhttpopen",
    "winhttpconnect",
    "winhttpsendrequest",
    "wsastartup"
]

In [ ]:
#Count networking APIs per file:
def count_matching_apis(api_names, search_terms):
    return sum(
        any(term in api_name for term in search_terms)
        for api_name in api_names
    )


network_df = pd.DataFrame({
    "network_api_count": api_lists.apply(
        lambda names: count_matching_apis(names, network_terms)
    ),
    "label": train_df["label"]
})

network_df["category"] = network_df["label"].map({
    0: "Benign",
    1: "Malware"
})

In [ ]:
#Calculate how many files import at least one networking API:
network_summary = (
    network_df.assign(
        uses_network_api=network_df["network_api_count"] > 0
    )
    .groupby("category")
    .agg(
        files=("uses_network_api", "size"),
        files_using_networking=("uses_network_api", "sum"),
        average_network_apis=("network_api_count", "mean"),
        median_network_apis=("network_api_count", "median")
    )
    .reset_index()
)

network_summary["percentage_using_networking"] = (
    network_summary["files_using_networking"]
    / network_summary["files"]
    * 100
)

display(network_summary.round(2))

In [ ]:
#Visualize the percentage:
alt.Chart(network_summary).mark_bar().encode(
    x=alt.X(
        "category:N",
        title="File type"
    ),
    y=alt.Y(
        "percentage_using_networking:Q",
        title="Files importing networking APIs (%)"
    ),
    tooltip=[
        "category:N",
        alt.Tooltip(
            "percentage_using_networking:Q",
            title="Percentage",
            format=".2f"
        ),
        "files_using_networking:Q",
        "files:Q"
    ]
).properties(
    title="Networking API Usage: Malware vs. Benign",
    width=500,
    height=350
)

In [ ]:
"""Compare several API categories"""
api_categories = {
    "Networking": [
        "socket", "connect", "send", "recv",
        "internetopen", "httpsendrequest",
        "winhttp", "wsastartup"
    ],

    "File operations": [
        "createfile", "readfile", "writefile",
        "deletefile", "copyfile", "movefile"
    ],

    "Registry": [
        "regopenkey", "regsetvalue", "regqueryvalue",
        "regcreatekey", "regdeletekey"
    ],

    "Process and memory": [
        "createprocess", "openprocess",
        "virtualalloc", "virtualprotect",
        "writeprocessmemory", "createremotethread"
    ],

    "Cryptography": [
        "cryptencrypt", "cryptdecrypt",
        "cryptacquirecontext", "bcrypt",
        "certopenstore"
    ]
}

In [ ]:
#Create category-level data:
category_rows = []

for category_name, terms in api_categories.items():
    counts = api_lists.apply(
        lambda names: count_matching_apis(names, terms)
    )

    for file_type, label_value in [
        ("Benign", 0),
        ("Malware", 1)
    ]:
        class_counts = counts[train_df["label"] == label_value]

        category_rows.append({
            "api_category": category_name,
            "file_type": file_type,
            "percentage_of_files": (
                (class_counts > 0).mean() * 100
            ),
            "average_count": class_counts.mean()
        })

api_category_df = pd.DataFrame(category_rows)

display(api_category_df.round(2))

In [ ]:
# Visualize:
alt.Chart(api_category_df).mark_bar().encode(
    x=alt.X(
        "api_category:N",
        title="API category"
    ),
    xOffset="file_type:N",
    y=alt.Y(
        "percentage_of_files:Q",
        title="Files importing category (%)"
    ),
    color=alt.Color(
        "file_type:N",
        title="File type"
    ),
    tooltip=[
        "api_category:N",
        "file_type:N",
        alt.Tooltip(
            "percentage_of_files:Q",
            title="Percentage",
            format=".2f"
        ),
        alt.Tooltip(
            "average_count:Q",
            title="Average API count",
            format=".2f"
        )
    ]
).properties(
    title="Imported API Categories: Malware vs. Benign",
    width=700,
    height=400
)

## Behavior TRAIN tag frequency

`win32_behavior_train_20pct.parquet` tags each sample with 0+ ClarAVy behavior labels (e.g. `backdoor`, `worm`, `downloader`). ClarAVy uses an empty string as a placeholder for "no confident tag," so that's filtered out before counting.

In [ ]:


behavior_df = pd.read_parquet(DATA_DIR / "win32_behavior_train_20pct.parquet")

print("rows:", len(behavior_df))
display(behavior_df.head())

In [ ]:
#Flatten the behavior tag lists, dropping the empty-string "no tag" placeholder
behavior_tags = pd.Series(
    [tag for tags in behavior_df["behavior"] for tag in tags if tag]
)

tag_counts = (
    behavior_tags
    .value_counts()
    .rename_axis("behavior_tag")
    .reset_index(name="count")
)

print("unique behavior tags:", len(tag_counts))
display(tag_counts.head(10))

In [ ]:
#Horizontal bar chart of the top behavior tags by frequency
TOP_N = 15
top_tags = tag_counts.head(TOP_N)

alt.Chart(top_tags).mark_bar(
    color="#2a78d6",
    cornerRadiusEnd=4
).encode(
    x=alt.X("count:Q", title="Number of Samples"),
    y=alt.Y("behavior_tag:N", title="Behavior Tag", sort="-x"),
    tooltip=[
        alt.Tooltip("behavior_tag:N", title="Behavior tag"),
        alt.Tooltip("count:Q", title="Samples", format=",")
    ]
).properties(
    title=f"Top {TOP_N} Malware Behavior Tags",
    width=650,
    height=400
)

## Common co-occurring behavior tag pairs

Since `behavior` is multi-label, some samples carry 2+ tags (e.g. a backdoor that also spies on the user). This counts, per sample, every unordered pair of distinct real tags it carries, then ranks pairs by how often they co-occur.

In [ ]:
from itertools import combinations
from collections import Counter

#Count each unordered pair of distinct real tags that co-occurs within a sample
pair_counts = Counter()
for tags in behavior_df["behavior"]:
    unique_tags = sorted(set(t for t in tags if t))
    for pair in combinations(unique_tags, 2):
        pair_counts[pair] += 1

pair_df = pd.DataFrame(
    [(f"{a} + {b}", count) for (a, b), count in pair_counts.items()],
    columns=["tag_pair", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

print("unique co-occurring pairs:", len(pair_df))
display(pair_df.head(10))

In [ ]:
#Horizontal bar chart of the top co-occurring behavior tag pairs
TOP_N_PAIRS = 15
top_pairs = pair_df.head(TOP_N_PAIRS)

alt.Chart(top_pairs).mark_bar(
    color="#2a78d6",
    cornerRadiusEnd=4
).encode(
    x=alt.X("count:Q", title="Number of Samples"),
    y=alt.Y("tag_pair:N", title="Behavior Tag Pair", sort="-x"),
    tooltip=[
        alt.Tooltip("tag_pair:N", title="Tag pair"),
        alt.Tooltip("count:Q", title="Samples", format=",")
    ]
).properties(
    title=f"Top {TOP_N_PAIRS} Co-occurring Behavior Tag Pairs",
    width=650,
    height=400
)

VISUALIZATION FOR DETECTION TEST

Malware vs Benign distribution

# We examine the detection test-set class balance to confirm that benign and
# malware samples are represented fairly. A balanced test set makes evaluation
# metrics such as accuracy, precision, and recall easier to interpret.

In [ ]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"

test_df = pd.read_parquet(
    DATA_DIR / "win32_test_detection.parquet"
)

print("Detection test shape:", test_df.shape)

In [ ]:
import altair as alt

alt.renderers.enable("default")
alt.data_transformers.enable("default")

In [ ]:
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

display(test_df.head())

In [ ]:
test_label_counts = (
    test_df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

test_label_counts["category"] = test_label_counts["label"].map({
    0: "Benign",
    1: "Malware"
})

test_label_counts["percentage"] = (
    test_label_counts["count"]
    / test_label_counts["count"].sum()
    * 100
)

display(test_label_counts)

In [ ]:
import altair as alt

bars = (
    alt.Chart(test_label_counts)
    .mark_bar()
    .encode(
        x=alt.X(
            "category:N",
            title="File classification",
            sort=["Benign", "Malware"]
        ),
        y=alt.Y(
            "count:Q",
            title="Number of samples"
        ),
        color=alt.Color(
            "category:N",
            legend=None
        ),
        tooltip=[
            alt.Tooltip("category:N", title="Classification"),
            alt.Tooltip("count:Q", title="Samples", format=","),
            alt.Tooltip("percentage:Q", title="Percentage", format=".2f")
        ]
    )
)

labels = (
    alt.Chart(test_label_counts)
    .mark_text(
        dy=-10,
        fontSize=13
    )
    .encode(
        x=alt.X(
            "category:N",
            sort=["Benign", "Malware"]
        ),
        y="count:Q",
        text=alt.Text(
            "count:Q",
            format=","
        )
    )
)

test_class_balance_chart = (
    bars + labels
).properties(
    title="Detection Test Set: Malware vs. Benign Distribution",
    width=500,
    height=350
)

test_class_balance_chart

Train vs Test Behavior Distribution

In [ ]:
import json
from pathlib import Path

import altair as alt
import pandas as pd


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    ROOT = CURRENT_DIR.parent
else:
    ROOT = CURRENT_DIR

DATA_DIR = ROOT / "win32_data"

behavior_train_df = pd.read_parquet(
    DATA_DIR / "win32_behavior_train_20pct.parquet"
)

behavior_test_df = pd.read_parquet(
    DATA_DIR / "win32_test_behavior.parquet"
)

print("Behavior train shape:", behavior_train_df.shape)
print("Behavior test shape:", behavior_test_df.shape)

print("\nTrain columns:", behavior_train_df.columns.tolist())
print("Test columns:", behavior_test_df.columns.tolist())

In [ ]:
print("Train behavior example:")
print(behavior_train_df.iloc[0]["behavior"])

print("\nTest behavior example:")
print(behavior_test_df.iloc[0]["behavior"])

print("\nTrain behavior type:")
print(type(behavior_train_df.iloc[0]["behavior"]))

In [ ]:
import json
import numpy as np
import pandas as pd


def parse_json(value):
    if isinstance(value, (dict, list, np.ndarray)):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return value

    return value


def extract_behaviors(value):
    parsed_value = parse_json(value)

    if parsed_value is None:
        return []

    if isinstance(parsed_value, float) and pd.isna(parsed_value):
        return []

    # Convert NumPy arrays into regular Python lists.
    if isinstance(parsed_value, np.ndarray):
        parsed_value = parsed_value.tolist()

    if isinstance(parsed_value, list):
        behaviors = []

        for item in parsed_value:
            if isinstance(item, str):
                item = item.strip()

                if item:
                    behaviors.append(item)

            elif isinstance(item, dict):
                behavior_name = (
                    item.get("name")
                    or item.get("behavior")
                    or item.get("description")
                    or item.get("label")
                    or item.get("value")
                )

                if behavior_name:
                    behaviors.append(str(behavior_name))

        return behaviors

    if isinstance(parsed_value, str):
        parsed_value = parsed_value.strip()

        if parsed_value:
            return [parsed_value]

    if isinstance(parsed_value, dict):
        return [str(key) for key in parsed_value.keys()]

    return []

In [ ]:
sample_behaviors = extract_behaviors(
    behavior_train_df.iloc[0]["behavior"]
)

print(sample_behaviors)

In [ ]:
behavior_train_df["extracted_behaviors"] = (
    behavior_train_df["behavior"]
    .apply(extract_behaviors)
)

display(
    behavior_train_df[
        ["behavior", "extracted_behaviors"]
    ].head(10)
)

In [ ]:
train_behavior_long = (
    behavior_train_df["extracted_behaviors"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

train_behavior_long = train_behavior_long[
    train_behavior_long != ""
]

print("Total extracted behavior records:", len(train_behavior_long))
print("Unique behaviors:", train_behavior_long.nunique())

print("\nMost common behaviors:")
print(train_behavior_long.value_counts().head(15))

In [ ]:
behavior_test_df["extracted_behaviors"] = (
    behavior_test_df["behavior"]
    .apply(extract_behaviors)
)

test_behavior_long = (
    behavior_test_df["extracted_behaviors"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

test_behavior_long = test_behavior_long[
    test_behavior_long != ""
]

print("Total test behavior records:", len(test_behavior_long))
print("Unique test behaviors:", test_behavior_long.nunique())

print("\nMost common test behaviors:")
print(test_behavior_long.value_counts().head(15))

In [ ]:
train_behavior_counts = (
    train_behavior_long
    .value_counts()
    .rename_axis("behavior")
    .reset_index(name="count")
)

train_behavior_counts["dataset"] = "Train"

test_behavior_counts = (
    test_behavior_long
    .value_counts()
    .rename_axis("behavior")
    .reset_index(name="count")
)

test_behavior_counts["dataset"] = "Test"

In [ ]:
train_behavior_counts["percentage"] = (
    train_behavior_counts["count"]
    / train_behavior_counts["count"].sum()
    * 100
)

test_behavior_counts["percentage"] = (
    test_behavior_counts["count"]
    / test_behavior_counts["count"].sum()
    * 100
)

In [ ]:
top_behaviors = (
    train_behavior_counts
    .head(15)["behavior"]
    .tolist()
)

print(top_behaviors)

In [ ]:
behavior_comparison_df = pd.concat(
    [
        train_behavior_counts[
            train_behavior_counts["behavior"].isin(top_behaviors)
        ],
        test_behavior_counts[
            test_behavior_counts["behavior"].isin(top_behaviors)
        ]
    ],
    ignore_index=True
)

In [ ]:
complete_index = pd.MultiIndex.from_product(
    [
        top_behaviors,
        ["Train", "Test"]
    ],
    names=["behavior", "dataset"]
)

behavior_comparison_df = (
    behavior_comparison_df
    .set_index(["behavior", "dataset"])
    .reindex(complete_index, fill_value=0)
    .reset_index()
)

display(behavior_comparison_df)

In [ ]:
import altair as alt

behavior_distribution_chart = (
    alt.Chart(behavior_comparison_df)
    .mark_bar()
    .encode(
        y=alt.Y(
            "behavior:N",
            title="Behavior",
            sort=top_behaviors
        ),
        x=alt.X(
            "percentage:Q",
            title="Percentage of behavior records"
        ),
        yOffset=alt.YOffset(
            "dataset:N"
        ),
        color=alt.Color(
            "dataset:N",
            title="Dataset"
        ),
        tooltip=[
            alt.Tooltip(
                "behavior:N",
                title="Behavior"
            ),
            alt.Tooltip(
                "dataset:N",
                title="Dataset"
            ),
            alt.Tooltip(
                "count:Q",
                title="Occurrences",
                format=","
            ),
            alt.Tooltip(
                "percentage:Q",
                title="Percentage",
                format=".2f"
            )
        ]
    )
    .properties(
        title="Train vs. Test Behavior Distribution",
        width=650,
        height=500
    )
)

behavior_distribution_chart